# Theory 
Joint Army Navy Air Force (JANAF) polynomials
https://webbook.nist.gov/cgi/cbook.cgi?ID=C1317608&Units=SI&Mask=2&Type=JANAFS&Table=on#JANAFS  
$ p_\mathrm{std} = 1~\mathrm{bar} $  
$ T_\mathrm{std} = 298.15~\mathrm{K} $  

## NASA-7  
Heat capacity [eqn. (1), 3]: $$\frac{C_p^o(T)}{R} = a_1 + a_2 T + a_3 T^2 + a_4 T^3 + a_5 T^4 $$  
Enthalpy [p.9, 3]: $$\frac{H^o(T)}{RT} = a_1 + a_2 \frac{T}{2} + a_3 \frac{T^2}{3} + a_4 \frac{T^3}{4} + a_5 \frac{T^4}{5} + \frac{b_1}{T} $$  
Entropy: $$\frac{S^o(T)}{R} = a_1 ln(T) + a_2 T + a_3 \frac{T^2}{2} + a_4 \frac{T^3}{3} + a_5 \frac{T^4}{4} + b_2 $$  
  
Reference enthalpy for assigned elements is  
$$ \Delta_f H^o(298.15) = H^o(298.15) = 0. $$  
  
Reference enthalpy for all species is  
$$ H^o(298.15) = \Delta_f H^o(298.15). $$  
  
Thus, the enthalpy is calculated as  
$$ H^o(T) = H^o(298.15) + \{ H^o(T) - H^o(298.15) \}, $$  
where $\{ H^o(T) - H^o(298.15) \}$ is the sensible enthalpy.

1. McBride, Bonnie J. Coefficients for calculating thermodynamic and transport properties of individual species. Vol. 4513. National Aeronautics and Space Administration, Office of Management, Scientific and Technical Information Program, 1993.
2. (not used) Chase, Malcolm W. "NIST-JANAF Thermochemical Tables 4th ed." J. of Physical and Chemical Reffernce Data (1998): 1529-1564.
2. http://combustion.berkeley.edu/gri-mech/data/nasa_plnm.html
3. http://akrmys.com/thermNASA/NASApolynomial.pdf (NASA7 (before 1994) and NASA9 (from 1994) difference)



In [ ]:
from os.path import join
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

mm = 1.0/2.54/10

repo_root_path = '.'
python_modules_path = join(repo_root_path, 'src', 'python')
if python_modules_path not in sys.path:
    sys.path.append(python_modules_path)

from specie_properties import janaf
from scm.scm import IlmenitePhase, ilmenite, ilmenite_orig, const_MW_Fe2O3, const_MW_FeO, const_MW_FeO
from specie_properties.janaf_ilmenite import janafs_ilmenite, janafs_ilmenite_orig, _get_McBride_janaf

In [ ]:
# Test

specie = 'Ar'
import cantera as ct
gas = ct.Solution('gri30.yaml')
MW = gas.molecular_weights[gas.species_index(specie)]
print(f"molWeight       {MW};")

j7_gri30 = janaf.janaf7Cantera(specie)
j7_gri30.print_coeffs_openfoam()

In [ ]:
jnw = janaf.janafNistWeb('N2')
jnw._coeff_dicts

# NIST vs GRI 3.0 JANAF

In [ ]:
def plot_comparison(ax, objs, args, styles, param):
    for object, style in zip(objs, styles):
        func = getattr(object, param)
        ax.plot(args, [func(arg) for arg in args], **style)
    ax.legend()

def plot_multiple_comparisons(axs, objs, Ts, styles, params):
    for ax, param in zip(axs, params):
        plot_comparison(ax, objs, Ts, styles, param)

# Test

specie = 'H2'

species = ['N2','O2','CO','CO2','H2O','H2','CH4','AR']
# species = ['H2']

# j7 = janaf7()
# j7.add_coeffs([3.53100528e+00, -1.23660987e-04, -5.0299437e-07, 2.43530612e-09, -1.40881235e-12, -1.046976288e+03, 2.96747468e+00], 200, 1000) # N2 [1, p.39(43)]
# j7.add_coeffs([2.95257626E+00, 1.39690057E-03, -4.92631691E-07, 7.86010367E-11, -4.60755321E-15, -9.23948645E+02, 5.87189252E+00], 1000, 6000)

for specie in species:

    jnw = janaf.janafNistWeb(specie)
    j7_gri30_cantera = janaf.janaf7Cantera(specie)

    Ts = np.arange(300,1500,20)
    janafs = [jnw, j7_gri30_cantera]
    styles = [
        dict(label='Nist website'),
        # dict(label='McBride 1993 7 coeff'),
        # dict(label='GRI3.0 7 coeff'),
        dict(label='GRI3.0 7 coeff cantera'),
    ]
    fig, axs = plt.subplots(ncols=3, figsize=[14,4])

    plot_multiple_comparisons(axs, janafs, Ts, styles, ['Cp0', 'H0', 'S0'])
    axs[0].set(xlabel='T,K', ylabel='Cp, J/kg/K')
    axs[1].set(xlabel='T,K', ylabel='H0, J/kg')
    axs[2].set(xlabel='T,K', ylabel='S0, J/kg/K')

    fig.suptitle(specie)
    fig.tight_layout()
    # fig.savefig(f'thermo_{specie}.png', dpi=300)


# Reaction heat correction
## Enthalpy of formation verification
Check if enthalpy of formation from table corresponds to the one calculated with JANAF
In JANAF it corresponds to enthalpy at 298.15 K

In [ ]:
R_McBride = 8.314463

dHfs_over_R = { # enthalpies of formation taken from McBride's book (last entries in coeffs)
    'Fe2O3': -9.92620367E+04,
    'FeO': -3.27183475E+04,
    'TiO2': -1.13628959E+05,
    'Fe3O4': -1.34696136E+05,
    'CaS': -5.69152785E+04,
    'CaSO4': -1.72486926E+05,
    # 'NiO': 3.77657540E+04,
    # 'Ni': 5.17319098E+04
}

for sp, dHf_over_R in dHfs_over_R.items():
    dHf_table = dHf_over_R*R_McBride
    # dHf_janaf = janafs_ilmenite_orig[sp].dH0f()
    dHf_janaf = _get_McBride_janaf(sp).dH0f()
    err = 100*(dHf_table-dHf_janaf)/dHf_table
    print(f'dH0f_{sp:5} ERR = {err:.2e} %')

# Reaction heat calculation for other oxygen carriers  
To get an idea of what others use

In [ ]:
T = 298.15
# T = 273.15 + 950

print("CaSO4 + 4H2 -> CaS + 4H2O")
print(  _get_McBride_janaf('CaS').H0(T) 
      + 4*janaf.janaf7Cantera('H2O').H0(T) 
      - _get_McBride_janaf('CaSO4').H0(T) 
      - 4*janaf.janaf7Cantera('H2').H0(T))

# print("CH4 + 4NiO -> 4Ni + 2H2O + CO2")
# dH =  ( 0*4*_get_McBride_janaf('Ni').H0(T) 
#       + 2*janaf.janaf7Cantera('H2O').H0(T) 
#       +   janaf.janaf7Cantera('CO2').H0(T) 
#       - 4*_get_McBride_janaf('NiO').H0(T) 
#       - janaf.janaf7Cantera('CH4').H0(T))
# print(dH)

## Validation of reaction heat correction

In [ ]:
janafs_ilmenite_orig['FeO'].dH0f()/(const_MW_FeO/1000)

In [ ]:
janafs_ilmenite_orig['Fe2O3'].dH0f()/(const_MW_Fe2O3/1000)

In [ ]:
#   O2 + 4 FeO -> 2 Fe2O3
# a O2 + b FeO -> d Fe2O3
a=1; b=4; d=2

# Hallberg et al. - A method for determination of reaction enthalpy of oxygen carriers for chemical looping combustion – Application to ilmenite, 2011
dHRm_ox_paper = -468500 # [J/mol O2]
T_paper = 925 + 273.15
# T_paper = 298.15

dHRm_ox_orig = d*janafs_ilmenite_orig['Fe2O3'].H0(T_paper) \
             - b*janafs_ilmenite_orig['FeO'].H0(T_paper) \
             - a*janaf.janaf7Cantera('O2').H0(T_paper) # [kJ/mol O2]

dHRm_ox_corrected = d*janafs_ilmenite['Fe2O3'].H0(T_paper) \
                  - b*janafs_ilmenite['FeO'].H0(T_paper) \
                  - a*janaf.janaf7Cantera('O2').H0(T_paper) # [kJ/mol O2]

print(f"Heat of reaction from the paper: {dHRm_ox_paper/1000:10.2f} kJ/mol O2")
print(f"Heat of reaction based on janaf: {dHRm_ox_orig/1000 :10.2f} kJ/mol O2")
print(f"Heat of reaction corrected:      {dHRm_ox_corrected/1000 :10.2f} kJ/mol O2")

print()
print(f"Original Hf_298.15 of Fe2O3:     {janafs_ilmenite_orig['Fe2O3'].dH0f()/1000 :10.2f} kJ/mol Fe2O3")
print(f"Corrected Hf_298.15 of Fe2O3:    {janafs_ilmenite['Fe2O3'].dH0f()/1000 :10.2f} kJ/mol Fe2O3")
print()
print(f"Original Hf_{T_paper:.2f} of Fe2O3:    {janafs_ilmenite_orig['Fe2O3'].H0(T_paper)/1000 :10.2f} kJ/mol Fe2O3")
print(f"Corrected Hf_{T_paper:.2f} of Fe2O3:   {janafs_ilmenite['Fe2O3'].H0(T_paper)/1000 :10.2f} kJ/mol Fe2O3")
print()

print("Updated janaf for Fe2O3")
janafs_ilmenite['Fe2O3'].print_coeffs_openfoam()
print("Unchanged janaf for FeO")
janafs_ilmenite['FeO'].print_coeffs_openfoam()

# Heat capacity
Using `checkThermo -phase solids` from OFUtils (in-house code)

In [ ]:
import os

specie = 'Fe2O3.solids'
in_section = False
buffer = []
if os.path.exists('run/0D/0D_act_CH4/Cpvalues.txt'):
    for line in open('run/0D/0D_act_CH4/Cpvalues.txt', 'r'):
        if line.startswith(specie):
            in_section = True
        elif line.strip() == '':
            in_section = False
        if in_section:
            # print(line.replace('K: ', '').strip())
            buffer.append(line.replace('K: ', '').strip())

    dat = np.loadtxt(buffer, skiprows=1, unpack=True)
    fig, ax = plt.subplots()
    ax.plot(dat[0], dat[1], '-o', label='OF')
    Ts = np.arange(300, 2500, 20)
    MW_Fe2O3 = 159.687
    ax.plot(Ts, [janafs_ilmenite['Fe2O3'].Cp0(T)/(MW_Fe2O3/1000) for T in Ts], label='Corrected Fe2O3')
    # ax.plot(dfs["Fe2O3"]['Cp'], label='OF')
    ax.set(
        xlabel = r'$T$, K',
        ylabel = r'$c_\mathrm{p}, \frac{\mathrm{J}}{\mathrm{kg K}}$'
    )
    ax.legend()
else:
    print("File 'run/0D/0D_act_CH4/Cpvalues.txt' does not exist.")

In [ ]:
ip = IlmenitePhase(3000, 0.033)
ip.volFractions(0)

v = dict()
v["Fe2O3"], v["FeO"], v["inerts"] = ip.volFractions(0)
v

Run testThermo  
```
testThermo solids Fe2O3
testThermo solids FeO
testThermo solids inerts
```

In [ ]:
fig, ax = plt.subplots(figsize=(140*mm, 80*mm))

# def extract_value(lines, entry):
#     for line in lines:
#         if line.startswith(entry):
#             return float(line.split()[1][:-1])

chemistries = ["pre", "act"]
ilmenite_states = ["ox", "red"]

species = [
    "Fe2O3",
    "FeO",
    "inerts"
]
Y = dict()
dfs = dict()

for chemistry in chemistries:
    for ilmenite_state in ilmenite_states:
        # ICBC_filename = f"ICBC_ilmenite_{ilmenite_state}_{chemistry}"
        # with open('etc/caseDicts/'+ICBC_filename) as file:
        #     lines = [line.rstrip() for line in file]
        #     Y["Fe2O3"]  = extract_value(lines, "YFe2O3")
        #     Y["FeO"]    = extract_value(lines, "YFeO")
        #     Y["inerts"] = extract_value(lines, "Yi")
        X   = 1 if (ilmenite_state == "ox") else 0
        rho = 4100 if (chemistry == "pre") else 4250
        Ro =  0.04 if (chemistry == "pre") else 0.033
        ip = IlmenitePhase(rhoox=rho, Ro=Ro)
        Y["Fe2O3"], Y["FeO"], Y["inerts"] = ip.massFractions(X)

        # The following files were generated with e.g.
        # testThermo solids inerts
        dfs["Fe2O3"]  = pd.read_csv("run/0D/0D_act_CH4/Fe2O3.dat", header=0, sep='\s+', index_col=0)
        dfs["FeO"]    = pd.read_csv("run/0D/0D_act_CH4/FeO.dat", header=0, sep='\s+', index_col=0)
        dfs["inerts"] = pd.read_csv("run/0D/0D_act_CH4/inerts.dat", header=0, sep='\s+', index_col=0)
        # print(ICBC_filename)
        Cp = Y["Fe2O3"]*dfs["Fe2O3"]["Cp"] \
           + Y["FeO"]*dfs["FeO"]["Cp"] \
           + Y["inerts"]*dfs["inerts"]["Cp"]
        # print(f"Cp = {Cp}")
        ax.plot(Cp, label=f"{ilmenite_state}_{chemistry}")
Cpdat = np.loadtxt("data/ilmenite/heat_capacity.dat", unpack=True)
ax.scatter(Cpdat[0], 1000*Cpdat[1], s=10, marker='x', label="Anovitz et al. 1985")
def Cp(T): # Anovitz et al. 1985, Table 3
    return 1.73094-5.2713e-4*T+2.2325e-7*T*T-16.725*1/np.sqrt(T)+2522*1/(T*T)
Ts = np.arange(300,1051,10)
ax.plot(Ts, Cp(Ts)*1000, label="Smoothened Anovitz et al. 1985")
ax.axvspan(800, 1300, alpha=0.2, color='red')
ax.annotate(text='', xy=(950,895), xytext=(950,965), arrowprops=dict(arrowstyle='<->'))
ax.annotate("~8%", xy=(950, 930))
ax.legend()
ax.set(
    xlabel = r"$T$, K",
    ylabel = r"$c_\mathrm{p}, \frac{\mathrm{J}}{\mathrm{kg K}}$",
)

# Reaction heat

In [ ]:
def getj7(specie, janafs=janafs_ilmenite):
    if specie in janafs.keys():
        return janafs[specie]
    else:
        return janaf.janaf7Cantera(specie)

def h_corr(specie, T=None):
    if T is None:
        return getj7(specie).dH0f()
    else:
        return getj7(specie).H0(T)

def h_orig(specie, T=None):
    if T is None:
        return getj7(specie, janafs=janafs_ilmenite_orig).dH0f()
    else:
        return getj7(specie, janafs=janafs_ilmenite_orig).H0(T)

print('Negative means enthalpy of the system reduced and the heat was released')

TdegC = 925

### ORIGINAL JANAFS
print()
print('Per mole of gaseous reactant (consistent with NRR) at 298.15 K (using enthalpies of formation)')
for sp in ['CH4', 'CO', 'H2', 'O2']:
    HRR = ilmenite['act'][sp].heat_release_rate(h_orig)
    print(f'{sp:3} reaction heat: {HRR/1000: 8.1f} kJ/mol')

print()
print(f'Per mole of gaseous reactant (consistent with NRR) at {TdegC} °C')
for sp in ['CH4', 'CO', 'H2', 'O2']:
    HRR = ilmenite['act'][sp].heat_release_rate(lambda s: h_orig(s, T=TdegC + 273.15))
    print(f'{sp:3} reaction heat: {HRR/1000: 8.1f} kJ/mol')

### CORRECTED JANAFS
print()
print('Per mole of gaseous reactant (consistent with NRR) at 298.15 K (using enthalpies of formation)')
for sp in ['CH4', 'CO', 'H2', 'O2']:
    HRR = ilmenite['act'][sp].heat_release_rate(h_corr)
    print(f'{sp:3} reaction heat: {HRR/1000: 8.1f} kJ/mol')

print()
print(f'Per mole of gaseous reactant (consistent with NRR) at {TdegC} °C')
for sp in ['CH4', 'CO', 'H2', 'O2']:
    HRR = ilmenite['act'][sp].heat_release_rate(lambda s: h_corr(s, T=TdegC + 273.15))
    print(f'{sp:3} reaction heat: {HRR/1000: 8.1f} kJ/mol')

# 0d case calculation (outdated)

In [ ]:
cases = ["CH4", "CO", "H2", "O2"]

rho_gas = dict(CH4=0.2516, CO=0.2703, H2=0.2298, O2=0.2997)
rho_gas2 = dict(CH4=0.2516, CO=0.2703, H2=0.2298, O2=0.2829) # at T2
cp_gas = dict(CH4=1759, CO=1370, H2=1590.6, O2=1178.6)
cp_gas2 = dict(CH4=1759, CO=1370, H2=1590.6, O2=1188.5) # at T2
cp_gas = dict(CH4=1416.1, CO=1050.7, H2=1214.6, O2=900.3)
# rho_solids = 4250
cp_solids = 935

V_gas = 1
m_solids = 0.000050
Ro = 0.033
m_O2 = Ro*m_solids
N_O2 = m_O2/(31.998*0.001) # [mol]


C_gas = dict()
C = dict()
for case in cases:
    C_gas[case] = cp_gas[case]*rho_gas[case]*V_gas
    C_solids = cp_solids * m_solids
    C[case] = C_gas[case] + C_solids
    print(f"Cgas    = {C_gas[case]:.3f} J/K")
    print(f"Csolids = {C_solids:.3f} J/K")
    print(f"Ctotal  = {C[case]:.3f} J/K")

    
    print(f"reaction heat = {dHr[case]*N_O2:.3f} J")

    print(f"Change of temperature = {C[case]*dHr[case]*N_O2:.3f} K")

In [ ]:
cases = ["CH4", "CO", "H2", "O2"]

print("     dHr [kJ/mol_O2] dHr_theor [J]  dHE [J]  dTgas [K] dT_from_HE [K]")
for case in cases:
    dat_gas    = np.loadtxt(f"run/0D/0D_act_{case}/postProcessing/volIntegrateFn(fields=(h.gas),weightFields=(alpha.gasrho.gas))/0/volFieldValue.dat", unpack=True)
    dat_solids = np.loadtxt(f"run/0D/0D_act_{case}/postProcessing/volIntegrateFn(fields=(h.solids),weightFields=(alpha.solidsrho.solids))/0/volFieldValue.dat", unpack=True)
    dat_Tgas = np.loadtxt(f"run/0D/0D_act_{case}/postProcessing/probesFn/0/T.gas", unpack=True)
    dat_Tsolids = np.loadtxt(f"run/0D/0D_act_{case}/postProcessing/probesFn/0/T.solids", unpack=True)
    dhe_solids = dat_solids[1][-1]-dat_solids[1][0]
    dhe_gas = dat_gas[1][-1]-dat_gas[1][0]
    dTgas = dat_Tgas[1][-1]-dat_Tgas[1][0]
    dTsolids = dat_Tsolids[1][-1]-dat_Tsolids[1][0]
    # print(case)
    # print(f"{dhe_solids}")
    # print(f"{dhe_gas}")
    # print(f"{dhe_gas+dhe_solids}")
    
    print(f"{case:4}: {dHr[case]/1000:15.3f} {-dHr[case]*N_O2:10.3f} {dhe_gas+dhe_solids:8.3f}  {dTgas:7.3f} {(dhe_gas+dhe_solids)/C[case]:8.3f}")
    # print(f"{dTsolids}")



print('The difference in energy values we observe in the case coulb be due to:')
print('1. Numerical errors. Resulting energy difference can change even by 100% (H2 case, dt=1s->0.1s)')
print('2. Non-isobaric process. Heat capacity is not Cp')

     dHr [kJ/mol_O2] dHr_theor [J]  dHE [J]  dTgas [K] dT_from_HE [K]
CH4 :         134.432     -6.932   -3.466   -4.018   -8.599
CO  :         -48.731      2.513    5.008   18.852   15.141
H2  :          -7.577      0.391    0.779    8.774    2.392
O2  :        -468.495     24.158   24.901   72.894   78.659
The difference in energy values we observe in the case coulb be due to:
1. Numerical errors. Resulting energy difference can change even by 100% (H2 case, dt=1s->0.1s)
2. Non-isobaric process. Heat capacity is not Cp

# Misc. unfinished

In [ ]:
# Ihsan Barin Thermochemical Data of Pure Substances, p.I-27

def interplogKf(val1, val2, T1=1200, T2=1300, T=1223.15):
    return np.interp(T, [T1, T2], [val1, val2])

logKfFeO   = interplogKf(8.460,7.554)
logKfFe2O3 = interplogKf(22.250, 19.541)
logKfFe3O4 = interplogKf(31.878,28.217)
logKfFeTiO3 = interplogKf(40.555,36.428)
logKfTiO2 = interplogKf(31.655,28.504)
logKfH2O   = interplogKf(7.904,7.069)

# Fe2O3 + H2 -> 2*FeO + H2O
logK = 2*logKfFeO+logKfH2O-logKfFe2O3
# 3*Fe2O3 + H2 -> 2Fe3O4 + H2O
logK1 = 2*logKfFe3O4+logKfH2O-3*logKfFe2O3
# Fe3O4 + H2 -> 3*FeO + H2O
logK2 = 3*logKfFeO+logKfH2O-logKfFe3O4
print(logK, np.exp(logK))
print(logK1, np.exp(logK1))
print(logK2, np.exp(logK2))


In [ ]:
sds = speciesDataStorage()
sd = sds.get('Ar', verbose=True)

In [ ]:
df = pd.DataFrame(columns=['T[K]','Cp[J/K/mol]','S[J/K/mol]','-(G/H298)/T[J/K/mol]','H[kJ/mol]','H-H298[kJ/mol]','G[kJ/mol]','dHf[kJ/mol]','dGf[kJ/mol]','logKf[-]'])

In [ ]:
def janafdG(T, c):
    t = T/1000
    return janafdH(T, c) - T*janafS(T, c)
janafdG(300, sd.get_coeffs(300))

In [ ]:
import pandas as pd



def getBarinTable(specie, temperatures, num_format='%.3f'):
    # rows = []
    df = pd.DataFrame(columns=['T[K]','Cp[J/K/mol]','S[J/K/mol]','-(G/H298)/T[J/K/mol]','H[kJ/mol]','H-H298[kJ/mol]','G[kJ/mol]','dHf[kJ/mol]','dGf[kJ/mol]','logKf[-]'])
    for T in temperatures:
        Cp  = sd.janafCp(T)
        S   = sd.janafS(T)
        GHT = -(sd.janafG(T)-sd.get_coeffs(T)[7]*1e3)/T
        H   = sd.janafH(T)    /1e3
        HH298 = sd.janafdH(T)/1e3
        G   = sd.janafG(T)/1e3
        dHf = sd.get_coeffs(T)[7]
        dGf = sd.get_coeffs(T)[7]
        logKf = -sd.janafG(T)/8.314/T
        row = [T, Cp, S, GHT, H, HH298, G, dHf, dGf, logKf]
        df.loc[len(df.index)] = row
    return df


# T = 298
# sds.get('FeO').janafG(T)/8.314/T

pd.options.display.float_format = '{:,.3f}'.format
temperatures = [298.15]+list(range(300,1700,100))#+[1650]
df = getBarinTable('FeO', temperatures)
# pd.options.display.float_format
# print(df.to_string)
df